In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
import h5py
import numpy as np
import torch
import torch.nn.functional as F
import random
from torch_geometric.data import Data
from torch.utils.data import dataloader
from torch_geometric.nn import GCNConv, global_mean_pool
from torch.utils.data import TensorDataset, DataLoader

C:\ml4sci_gsoc_test\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
print(os.listdir())
graphs = torch.load("graphs_dataset.pt")

print("Loaded graphs:", len(graphs))
print(graphs[0])

['.ipynb_checkpoints', '01_data_and_autoencoder.ipynb', '02_graph_representation_gnn.ipynb', '02_improved_autoencoder.ipynb', '03_contrastive_learning.ipynb', 'graphs_dataset.pt']


C:\Users\Harsh Sharma\AppData\Local\Temp\ipykernel_12248\3178171671.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  graphs = torch.load("graphs_dataset.pt")


Loaded graphs: 8000
Data(x=[852, 5], edge_index=[2, 6816], y=[1])


# Creating the DataLoader of graphs

In [5]:
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

train_graphs, test_graphs = train_test_split(graphs, test_size=0.2 , random_state=42)
train_loader = DataLoader(train_graphs, batch_size=64, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=64,)
data = next(iter(train_loader))
print(type(data))

<class 'abc.DataBatch'>


# Graph Encoder (Reuse GNN)

In [6]:
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

import torch.nn.functional as F

class GraphEncoder(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(5, 64)
        self.bn1 = torch.nn.BatchNorm1d(64)

        self.conv2 = GCNConv(64, 128)
        self.bn2 = torch.nn.BatchNorm1d(128)

        self.conv3 = GCNConv(128, 128)
        self.bn3 = torch.nn.BatchNorm1d(128)

        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index, batch):

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = self.dropout(x)
        # pooling
        x_max = global_max_pool(x, batch)
        x_mean = global_mean_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)
        return x

# Projection Head

In [7]:
class ProjectionHead(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(256, 256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),

            torch.nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.net(x)

# Graph Augmentation

In [8]:
def augment_graph(data, drop_prob=0.1):

    x = data.x.clone()
    mask = torch.rand_like(x) > drop_prob
    x = x * mask.float()
    x = x + torch.randn_like(x) * 0.02
    return Data(
        x=x,
        edge_index=data.edge_index,
        y=data.y,
        batch=data.batch
    )

# Verify graph integrity

In [9]:
data = next(iter(train_loader))

print("Nodes:", data.x.shape)
print("Max edge index:", data.edge_index.max())
print("Num nodes:", data.x.shape[0])

Nodes: torch.Size([41623, 5])
Max edge index: tensor(41622)
Num nodes: 41623


# Contrastive Loss (Barlow loss)

In [10]:
def contrastive_loss(z1, z2):

    z1 = (z1 - z1.mean(0)) / (z1.std(0) + 1e-5)
    z2 = (z2 - z2.mean(0)) / (z2.std(0) + 1e-5)
    N, D = z1.size()
    c = torch.mm(z1.T, z2) / N
    I = torch.eye(D).to(z1.device)
    loss = ((c - I)**2).mean()
    return loss

# sanity tes

In [11]:
z1 = torch.randn(32, 128).to(torch.device('cuda'))
z2 = torch.randn(32, 128).to(torch.device('cuda'))
loss = contrastive_loss(z1, z2)
print(loss)

tensor(0.0377, device='cuda:0')


# Training Loop (Contrastive)

In [12]:
encoder = GraphEncoder().to(torch.device('cuda'))
projector = ProjectionHead().to(torch.device('cuda'))

In [13]:
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(projector.parameters()),
    lr=1e-3)
for epoch in range(5):
    total_loss = 0
    for data in train_loader:
        data = data.to(torch.device('cuda'))
        g1 = augment_graph(data)
        g2 = augment_graph(data)
        z1 = projector(encoder(g1.x, g1.edge_index, g1.batch))
        z2 = projector(encoder(g2.x, g2.edge_index, g2.batch))
        loss = contrastive_loss(z1, z2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss {total_loss/len(train_loader):.4f}")

Epoch 1, Loss 0.0244
Epoch 2, Loss 0.0201
Epoch 3, Loss 0.0197
Epoch 4, Loss 0.0199
Epoch 5, Loss 0.0196


In [14]:
emb = encoder(data.x, data.edge_index, data.batch)
print("Embedding std:", emb.std().item())

Embedding std: 1.7948342561721802


# Freeze encoder

In [87]:
for param in encoder.parameters():
    param.requires_grad = False

# Train classifier

In [76]:
class Classifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(128, 64),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            # torch.nn.Dropout(0.3),
            torch.nn.Linear(64, 32),
            torch.nn.BatchNorm1d(32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.net(x)
classifier = Classifier().to(torch.device('cuda'))

In [78]:
encoder.eval()
classifier.train()
optimizer = torch.optim.Adam(classifier.parameters(), lr=2e-3  )
for epoch in range(10):
    total_loss = 0
    for data in train_loader:
        data = data.to(torch.device('cuda'))
        with torch.no_grad():
            emb = encoder(data.x, data.edge_index, data.batch)
            emb = emb * 5
        out = classifier(emb)
        loss = F.cross_entropy(out, data.y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print("Embedding std:", emb.std().item())
    print(f"[Frozen Encoder] Epoch {epoch+1}, Loss {total_loss/len(train_loader):.4f}")

Embedding std: 0.1643555462360382
[Frozen Encoder] Epoch 1, Loss 0.6912
Embedding std: 0.18680591881275177
[Frozen Encoder] Epoch 2, Loss 0.6894
Embedding std: 0.1673414260149002
[Frozen Encoder] Epoch 3, Loss 0.6885
Embedding std: 0.17059925198554993
[Frozen Encoder] Epoch 4, Loss 0.6896
Embedding std: 0.19037114083766937
[Frozen Encoder] Epoch 5, Loss 0.6892
Embedding std: 0.1777515709400177
[Frozen Encoder] Epoch 6, Loss 0.6873
Embedding std: 0.16170184314250946
[Frozen Encoder] Epoch 7, Loss 0.6865
Embedding std: 0.15202908217906952
[Frozen Encoder] Epoch 8, Loss 0.6862
Embedding std: 0.18235830962657928
[Frozen Encoder] Epoch 9, Loss 0.6852
Embedding std: 0.17210248112678528
[Frozen Encoder] Epoch 10, Loss 0.6853


In [ ]:
for param in encoder.parameters():
    param.requires_grad = True

In [ ]:
encoder.train()
classifier.train()

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(classifier.parameters()),
    lr=5e-4   # higher than before
)
for epoch in range(20):
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        emb = encoder(data.x, data.edge_index, data.batch)
        # Add dropout (NEW)
        emb = F.dropout(emb, p=0.3, training=True)
        out = classifier(emb)
        loss = F.cross_entropy(out, data.y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[Fine-tune] Epoch {epoch+1}, Loss {total_loss/len(train_loader):.4f}")

In [56]:
from sklearn.metrics import roc_auc_score
encoder.eval()
preds= []
labels = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(torch.device('cuda'))
        out = encoder(data.x, data.edge_index, data.batch)
        prob =torch.softmax(out , dim = 1)[:,1]
        preds.extend(prob.cpu().numpy())
        labels.extend(data.y.cpu().numpy())
auc = roc_auc_score(labels, preds)
print("ROC-AUC:", auc)

ROC-AUC: 0.4630958888232816
